In [1]:
from langchain_community.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage
import requests


/tmp/ipykernel_18726/3794927.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import tool


In [ ]:
### Conversionn Rate Tool

import os
import requests
from langchain_core.tools import tool

@tool(parse_docstring=True)
def get_conversion_rate(base_currency: str, target_currency: str) -> float:
    """Fetch the current exchange rate between two currencies.

    Use this tool whenever you need the live exchange rate from one currency to
    another, either because the user asked for the rate directly, or because you
    need a rate before converting an amount. Never guess or recall
    exchange rates from memory, because they change constantly.

    Args:
        base_currency: The 3-letter ISO 4217 code of the currency being converted FROM, in uppercase, e.g. "USD", "EUR", "PKR".
        target_currency: The 3-letter ISO 4217 code of the currency being converted TO, in uppercase, e.g. "GBP", "JPY", "PKR".

    Returns:
        The Conversion rate between two currencies
    """
    api_key = os.getenv("EXCHANGE_API_KEY")
    url = f"https://v6.exchangerate-api.com/v6/{api_key}/pair/{base_currency.upper()}/{target_currency.upper()}"

    response = requests.get(url)
    return response.json()["conversion_rate"]




### Currency Convertor Tool
@tool(parse_docstring=True)
def convert_currency(base_currency_value: float, conversion_rate: float) -> float:
    """Convert an amount of money from a base currency to a target currency using a known conversion rate.

    Use this tool whenever the user wants to know how much a given amount in one
    currency is worth in another currency. You MUST already have the conversion
    rate before calling this tool.Do not guess or invent the rate, and do not do the
    multiplication yourself.

    Args:
        base_currency_value: The amount of money in the base (source) currency, e.g. 250.0 for 250 EUR.
        conversion_rate: The exchange rate for 1 unit of the base currency expressed in the target currency, e.g. 0.865 for EUR to GBP. Obtain it from `get_conversion_factor`.

    Returns:
        The converted amount in the target currency as a float.
    """
    return base_currency_value * conversion_rate

CurrencyConverter.invoke({"baseCurrencyValue":1,"ConversionRate":getConversionRate.invoke({"BaseCurrency":"USD","TargetCurrency":"PKR"})})

277.0451

In [ ]:
### Tools Attaching to LLM

tools={"getConversionRate":getConversionRate
       ,"CurrencyConverter":CurrencyConverter}

llm=ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", temperature=0.7)

llm_with_tools=llm.bind_tools([getConversionRate,CurrencyConverter])

In [13]:
### Creating History List
messages=[HumanMessage("Convert the 56 USD to PkR")]

AIMessage(content=[], additional_kwargs={'function_call': {'name': 'getConversionRate', 'arguments': '{"BaseCurrency": "USD", "TargetCurrency": "PKR"}'}, '__gemini_function_call_thought_signatures__': {'call_252822': 'EnMKcQFpFH0TdG7ir1HHmszToyN10vrIkvXRdKvWwe3+obHKYqEXgxUEVNP++5OFNm1s9hW08YL7mxniUFoksHwG/0OPhbs8QZ/4XpGspdvPdXgqZNvw2VP96wDWbUIspTyOcLfnm+Xncmho5xxg0ahq2Awd'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a163ef-d635-7240-9b4e-a7a7acb74ec4-0', tool_calls=[{'name': 'getConversionRate', 'args': {'BaseCurrency': 'USD', 'TargetCurrency': 'PKR'}, 'id': 'call_252822', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 144, 'output_tokens': 25, 'total_tokens': 169, 'input_token_details': {'cache_read': 0}})